# Logical Functions Data Preparation and Pivot Summaries

**STA 532 2.0 Spreadsheet Modelling** · [Subject index](00_Subject_Index.ipynb)

Work through the original theory notes, calculation tables, simulations and exercises in this notebook.

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Write the rule in words first](#theory-01)
- [IF and nested decisions](#theory-02)
- [AND, OR and negation](#theory-03)
- [Type and missing-value pitfalls](#theory-04)
- [Derived quantities](#theory-05)
- [Conditional and grouped summaries](#theory-06)
- [Auditing the chain](#theory-07)
- [Reproducibility](#theory-08)

## Detailed theory · Logical functions and aggregation rules

<a id="theory-01"></a>

### Write the rule in words first
A logical formula encodes a classification or decision. Define the inputs, missing-value treatment, cutoffs and outputs before writing IF. A condition can be technically executable yet scientifically ambiguous if a threshold's meaning or units are missing.

<a id="theory-02"></a>

### IF and nested decisions
IF(condition, true result, false result) branches on a logical test. Nested conditions are evaluated in order, so broad earlier conditions can make later ones unreachable. A lookup table may be clearer than a long chain when categories are numerous and ordered. Validate every category and an out-of-range case.

<a id="theory-03"></a>

### AND, OR and negation
AND requires all tests, OR permits any, and NOT reverses a test. Parenthesize combinations explicitly. De Morgan's laws help check negation: not(A and B) equals not A or not B. In array-oriented Python, elementwise Boolean operators differ from the scalar and/or keywords; preserve row-by-row intent.

<a id="theory-04"></a>

### Type and missing-value pitfalls
Text-form numbers, spaces in labels and inconsistent date parsing can make comparisons fail. A missing value is not always false or zero. Use an explicit rule for incomplete records and report how many observations were eligible for a calculation.

<a id="theory-05"></a>

### Derived quantities
Revenue, cost and profit must have matching periods and units. Profit=revenue−cost, revenue margin=profit/revenue and markup=profit/cost answer different questions. Guard against zero denominators and distinguish a missing result from a real zero margin.

<a id="theory-06"></a>

### Conditional and grouped summaries
An IF-based classification can become a pivot category. Count eligible records, sum their values, and state the denominator of any proportion. Aggregate ratios from sums when the target is a combined rate. A count of rows and a count of nonmissing numeric entries can differ.

<a id="theory-07"></a>

### Auditing the chain
Show original fields, intermediate truth values, combined condition, chosen label and aggregate contribution. Check boundary values, contradictory rules and totals. This makes a model explainable: a user can identify which input changed the outcome instead of seeing only a final label.

<a id="theory-08"></a>

### Reproducibility
Keep stable input schemas, codebooks, formula definitions and a small set of hand-verifiable examples. Sorting, filtering or adding rows should not silently shift references or omit new data. The lab's row-by-row truth table is the verification model for both spreadsheet and Python implementations.

## Theory
Logical spreadsheet functions turn a stated rule into a value. IF chooses one of two results; AND requires all conditions; OR requires at least one. Check boundary values because greater-than and greater-than-or-equal give different classifications.

Represent each case by a row and each variable by a column. An identifier is not a quantitative measurement. Preserve missing values explicitly and distinguish an absent value from a genuine zero.

A pivot summary groups observations and calculates counts, sums or other aggregates. Denominators must match the question: revenue share uses total revenue; within-state profit margin is total state profit divided by total state revenue, rather than an unweighted average of row margins.

## Worked reasoning
If revenue is 100 and cost is 70, profit is 30 and revenue-based margin is 30%. A second row with revenue 1000 and profit 100 has a 10% margin; combined margin is $130/1100=11.82\%$, rather than 20%.

## Python calculation and simulation

Predict the output, run the cell, then change one input and explain its effect.

## Worked calculation · IF, AND and pivot summaries: intermediate calculation table

### Question and assumptions

Classify a row as eligible when profit>0 AND revenue≥250, and label profit≥100 as meeting the profit target.

### Procedure

1. Compute profit before the logical conditions.
2. Show the truth value of each condition.
3. Combine conditions and group the results.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Region | Revenue | Cost | Profit | Profit>0 | Revenue≥250 | AND result | IF label |
| --- | --- | --- | --- | --- | --- | --- | --- |
| A | 100 | 70 | 30 | True | False | False | Below |
| A | 1000 | 900 | 100 | True | True | True | Target |
| B | 250 | 150 | 100 | True | True | True | Target |
| B | 400 | 280 | 120 | True | True | True | Target |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


data=[["A",100,70],["A",1000,900],["B",250,150],["B",400,280]]
rows=[]
for g,r,c in data:
    profit=r-c; rows.append([g,r,c,profit,profit>0,r>=250,(profit>0 and r>=250),"Target" if profit>=100 else "Below"])
show_table(["Region","Revenue","Cost","Profit","Profit>0","Revenue≥250","AND result","IF label"],rows)
assert sum(r[6] for r in rows)==3


### Decision and interpretation

Boundary rows matter: revenue equal to 250 satisfies ≥250, and profit equal to 100 satisfies ≥100. Write those symbols explicitly before entering a formula.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
state=np.array(['A','A','B','B'])
revenue=np.array([100.,1000.,250.,400.]); cost=np.array([70.,900.,150.,280.])
profit=revenue-cost
label=np.where(profit>=100,'At least 100 profit','Below 100 profit')
eligible=(profit>0)&(revenue>=250)
print('IF equivalents:',label)
print('AND equivalent:',eligible)
for group in np.unique(state):
    use=state==group; r=revenue[use].sum(); p=profit[use].sum()
    print('State:',group,'Revenue:',r,'Profit:',p,'Margin:',p/r)


## Exercises

1. Which rows satisfy both profit > 0 and revenue >= 250?
2. Explain the difference between a sum and count pivot.
3. Why should state A's combined margin not be the mean of its two margins?

<details><summary>Answer sketches</summary>

1. Rows 2, 3 and 4. 2. A sum aggregates numerical amounts; a count measures records or nonmissing values according to the chosen convention. 3. The revenues differ; use a revenue-weighted margin of about 11.82%.
</details>